# sparse-cl — lưới train backbone

Sáu cấu hình × {không regularizer, EWC-DR λ=100}, backbone **fine-tune toàn bộ**.

| # | Projection | Head |
|:-:|---|---|
| 1 | none | Linear |
| 2 | none | MLP |
| 3 | frozen | Linear |
| 4 | frozen | MLP |
| 5 | learnable | Linear |
| 6 | learnable | MLP |

Khác bảng backbone đóng băng ở ba điểm: `batch_size 64` (ViT-B fine-tune ở 256 cần ~22 GiB),
`epochs 20` (fine-tune từ pretrained hội tụ nhanh hơn nhiều so với train head từ đầu),
và **cả sáu cấu hình đều chạy được EWC-DR** — backbone có 86 M tham số trôi qua mọi task,
nên không còn ô `n/a` như trước.

**Trên Kaggle**: Accelerator = `GPU T4 ×2` (đừng chọn P100 — không có tensor core, chậm
~2.5×), Internet = ON, upload `sparse-cl/` thành Dataset rồi Add vào notebook.
Nhớ **Save Version → Save & Run All**, đóng tab là mất kết quả.

In [ ]:
import glob, os, shutil, subprocess, sys
import torch

# Kaggle: chép repo từ Dataset ra /kaggle/working. Máy khác: đã đứng sẵn trong repo.
hits = sorted(glob.glob('/kaggle/input/**/train.py', recursive=True), key=len)
if hits:
    os.chdir('/kaggle/working')
    shutil.rmtree('sparse-cl', ignore_errors=True)
    shutil.copytree(os.path.dirname(hits[0]), 'sparse-cl')
    os.chdir('sparse-cl')
assert os.path.exists('train.py'), 'khong tim thay train.py - dung thu muc chua?'

try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
    import timm

N_GPU = torch.cuda.device_count()
print('torch', torch.__version__, '| timm', timm.__version__)
print('GPU  ', [torch.cuda.get_device_name(i) for i in range(N_GPU)] or 'KHONG CO')
# bf16 can Ampere tro len; T4/P100 -> False, train.py tu chuyen sang fp16 + GradScaler
print('bf16 ', torch.cuda.is_bf16_supported() if N_GPU else '-')
assert N_GPU >= 1, 'Bat GPU: Settings > Accelerator > GPU'

## Chọn phần việc

Sửa ba biến trong cell dưới rồi chạy. Có 2 GPU thì tự chia đôi thành hai luồng song song.

`CONFIGS` nhận `a` (cấu hình 1,2,3), `b` (4,5,6), `all`, hoặc **danh sách số** như `'1,2'` / `'5'`.

Chia an toàn cho Kaggle — biên hơn 2× so với trần 12 h mỗi phiên:

| Notebook | `BACKBONE` | `CONFIGS` | `REGS` | Runs | T4 ×2 |
|---|---|---|---|---:|---:|
| NB1 | `vit` | `'1,2'` | `'both'` | 4 | ~5.6 h |
| NB2 | `vit` | `'3,4'` | `'both'` | 4 | ~5.6 h |
| NB3 | `vit` | `'5,6'` | `'both'` | 4 | ~5.6 h |
| NB4 | `resnet` | `'all'` | `'both'` | 12 | ~9 h |

Ước tính T4 là **ngoại suy, chưa đo**. Sau ~15 phút của run đầu, log in ra `train_time` của
task 0; nhân lên `train_time × 10 × số_run_mỗi_gpu × 1.1` là ra tổng thật. Nếu vượt 10 h thì
dừng và chia nhỏ thêm — Kaggle giết phiên ở 12 h và output chưa commit là mất trắng.

Máy riêng có GPU thì cứ để `'all'` + `'both'` (12 run mỗi backbone).

In [ ]:
import time

# ======================= SỬA Ở ĐÂY =======================
BACKBONE = 'vit'      # 'vit' | 'resnet'
CONFIGS  = '1,2'      # '1,2' | '5' | 'a' (cfg 1,2,3) | 'b' (cfg 4,5,6) | 'all'
REGS     = 'both'     # 'none' | 'ewc' | 'both'
# =========================================================

# Chia doi cho 2 GPU. Uu tien tach theo REGULARIZER vi hai nua luon bang nhau
# ve so run; tach theo cau hinh chi dung khi da chon 'all'. Cac luong ghi ra file
# JSON khac ten (exp_name ma hoa moi flag) nen khong dam nhau du chung thu muc.
if N_GPU >= 2 and REGS == 'both':
    jobs = [(0, CONFIGS, 'none'), (1, CONFIGS, 'ewc')]
elif N_GPU >= 2 and CONFIGS == 'all':
    jobs = [(0, 'a', REGS), (1, 'b', REGS)]
else:
    jobs = [(0, CONFIGS, REGS)]

procs, logs = [], []
for gpu, cfg, reg in jobs:
    log = f'log_gpu{gpu}.txt'
    cmd = ['bash', 'run_backbone.sh', BACKBONE, str(gpu), cfg, reg]
    print('->', ' '.join(cmd), '>', log)
    procs.append(subprocess.Popen(cmd, stdout=open(log, 'w'), stderr=subprocess.STDOUT))
    logs.append(log)

# Bam theo log de notebook co dau hieu song; in moi phut, chi cac dong dang ke.
# 'train_time' cua task 0 la thu dau tien can nhin: nhan x10 x so_run la ra tong.
t0, seen = time.time(), [0] * len(logs)
while any(p.poll() is None for p in procs):
    time.sleep(60)
    for i, log in enumerate(logs):
        lines = open(log, errors='ignore').read().splitlines()
        for ln in lines[seen[i]:]:
            if ln.startswith(('=====', 'XONG', 'Traceback')) or 'A_t=' in ln or 'A_bar' in ln:
                print(f'[{(time.time()-t0)/3600:5.2f}h gpu{i}] {ln[:120]}')
        seen[i] = len(lines)

print('\nma thoat:', [p.returncode for p in procs], f'| {(time.time()-t0)/3600:.2f} h')

## Kết quả

`A_T` = accuracy trên toàn bộ 100 lớp sau task cuối · `Ā` = trung bình tích luỹ qua 10 mốc ·
`Forgetting` = mức sụt trung bình từ đỉnh của mỗi task cũ (thấp hơn tốt hơn).

Hai cột chẩn đoán: `pen/clf` nên nằm trong 0.1–1 — nhỏ hơn thì regularizer vô hình,
lớn hơn thì mạng bị đóng băng. `best_ep` là epoch tốt nhất trung bình: **nếu nó sát 20
thì ngân sách epoch quá chặt, phải tăng `--epochs` rồi chạy lại**.

In [ ]:
import json

# Moi ket qua deu nam trong runs/ - exp_name da ma hoa moi flag anh huong so lieu
# nen khong can tach thu muc. Loc ra cac run co backbone train duoc.
rows = []
for f in sorted(glob.glob('runs/*.json')):
    d = json.load(open(f)); m, a, last = d['metrics'], d['args'], d['per_task'][-1]
    if a['freeze_backbone'] != 'False':
        continue
    proj = ('none' if a['expand_dim'] == '0'
            else 'learnable' if a['train_projection'] == 'True' else 'frozen')
    rows.append((a['model_name'].split('_')[0], proj,
                 'MLP' if a['use_mlp'] == 'True' else 'Linear',
                 'EWC-DR' if a['cl_reg'] != 'none' else '-',
                 m['A_T'], m['A_bar'], m['forgetting'],
                 last.get('pen_over_clf', '-'),
                 round(sum(p['best_epoch'] for p in d['per_task']) / len(d['per_task']), 1)))

hdr = ('backbone', 'projection', 'head', 'reg', 'A_T', 'A_bar', 'forget', 'pen/clf', 'best_ep')
w = (10, 11, 8, 8, 8, 8, 8, 9, 8)
print(''.join(f'{h:<{n}}' for h, n in zip(hdr, w)))
print('-' * sum(w))
for r in sorted(rows):
    print(''.join(f'{str(v):<{n}}' for v, n in zip(r, w)))
print('-' * sum(w))
print(f'{len(rows)} run backbone train duoc.')
print('Moc backbone DONG BANG (batch 256, 100 epoch, 3 seeds) - chi de tham chieu,')
print('dung so thang vi batch va so epoch deu khac; hay so voi dong frozen+Linear o tren:')
print('  ViT    frozen+Linear  A_T 83.17  A_bar 89.49  forget  7.49')
print('  ResNet frozen+Linear  A_T 62.56  A_bar 74.38  forget 14.77')

In [ ]:
# Dong goi de tai ve. Tren Kaggle nho Save Version, khong thi mat het.
shutil.make_archive(os.path.join(os.getcwd(), 'runs'), 'zip', '.', 'runs')
print('zip: runs.zip |', len(glob.glob('runs/*.json')), 'file JSON')